# Handling Class Imbalance
## Credit Card Fraud Detection Dataset

This notebook addresses the class imbalance problem in fraud detection datasets.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load datasets
card_trans_df = pd.read_csv('../Datasets/card_transdata.csv')
creditcard_df = pd.read_csv('../Datasets/creditcard.csv')

print("Loaded datasets successfully")

## 1. Analyze Class Distribution

In [ ]:
# Check class distribution for fraud column
def analyze_class_distribution(df, target_col):
    if target_col in df.columns:
        print(f"\nClass Distribution for {target_col}:")
        print(df[target_col].value_counts())
        print(f"\nClass Distribution (Percentage):")
        print(df[target_col].value_counts(normalize=True) * 100)
        return True
    else:
        print(f"Column {target_col} not found")
        print(f"Available columns: {df.columns.tolist()}")
        return False

# Try common fraud column names
fraud_cols = ['fraud', 'isFraud', 'Class', 'is_fraud', 'Fraud']

print("=" * 50)
print("Card Transaction Data:")
print("=" * 50)
found = False
for col in fraud_cols:
    if analyze_class_distribution(card_trans_df, col):
        found = True
        break

print("\n" + "=" * 50)
print("Credit Card Data:")
print("=" * 50)
found = False
for col in fraud_cols:
    if analyze_class_distribution(creditcard_df, col):
        found = True
        break

## 2. Handle Imbalance - Resampling Techniques

In [ ]:
# Method 1: Oversampling (Increase minority class)
def oversample_minority(df, target_col):
    """Oversample minority class to match majority class"""
    if target_col not in df.columns:
        return df
    
    majority = df[df[target_col] == 0]
    minority = df[df[target_col] == 1]
    
    # Oversample minority class
    minority_upsampled = resample(minority, 
                                   n_samples=len(majority),
                                   replace=True,
                                   random_state=42)
    
    # Combine
    balanced_df = pd.concat([majority, minority_upsampled])
    return balanced_df.sample(frac=1).reset_index(drop=True)

# Method 2: Undersampling (Decrease majority class)
def undersample_majority(df, target_col):
    """Undersample majority class to match minority class"""
    if target_col not in df.columns:
        return df
    
    majority = df[df[target_col] == 0]
    minority = df[df[target_col] == 1]
    
    # Undersample majority class
    majority_downsampled = resample(majority,
                                     n_samples=len(minority),
                                     replace=False,
                                     random_state=42)
    
    # Combine
    balanced_df = pd.concat([majority_downsampled, minority])
    return balanced_df.sample(frac=1).reset_index(drop=True)

print("Resampling functions created successfully")

In [ ]:
# Find the target column
def find_target_col(df):
    fraud_cols = ['fraud', 'isFraud', 'Class', 'is_fraud', 'Fraud']
    for col in fraud_cols:
        if col in df.columns:
            return col
    return None

# Apply resampling
card_trans_target = find_target_col(card_trans_df)
creditcard_target = find_target_col(creditcard_df)

if card_trans_target:
    card_trans_balanced = oversample_minority(card_trans_df, card_trans_target)
    print(f"Card Transaction Data - After Oversampling:")
    print(card_trans_balanced[card_trans_target].value_counts())
else:
    print("Card Transaction target column not found")

if creditcard_target:
    creditcard_balanced = oversample_minority(creditcard_df, creditcard_target)
    print(f"\nCredit Card Data - After Oversampling:")
    print(creditcard_balanced[creditcard_target].value_counts())
else:
    print("Credit Card target column not found")

## 3. Visualize Imbalance Before and After

In [ ]:
# Create visualizations comparing before and after resampling
# Add your visualization code here

## 4. Alternative: Use Class Weights During Model Training

Another approach to handle imbalance is to use class weights in the model:
- For sklearn models: use `class_weight='balanced'`
- For imbalanced-learn library: use SMOTE, ADASYN, etc.

In [ ]:
# Alternative: Use SMOTE (if imbalanced-learn is available)
# from imblearn.over_sampling import SMOTE
# 
# smote = SMOTE(random_state=42)
# X_resampled, y_resampled = smote.fit_resample(X, y)

print("Class imbalance handling completed")